## 2019-2023년 계절별 산사태 발생 이력에 따른 가중치 분배

In [9]:
import pandas as pd
import re

def extract_landslide_season_ratio(csv_path):
    """산사태 발생 CSV에서 계절별 발생 건수 및 비율을 정량적으로 추출"""

    # 인코딩 자동 감지
    for enc in ['cp949', 'euc-kr', 'utf-8-sig', 'utf-8']:
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            break
        except Exception:
            continue
    else:
        raise ValueError("❌ CSV 파일 인코딩 오류")

    # ① 날짜에서 월 추출
    def extract_month(datestr):
        match = re.search(r'(\d{4})-(\d{2})', str(datestr))
        if match:
            return int(match.group(2))
        return None

    df['발생월'] = df['재난구분'].apply(extract_month)

    # ② 월 → 계절 변환
    def get_season(month):
        if month in [3, 4, 5]: return 'spring'
        elif month in [6, 7, 8]: return 'summer'
        elif month in [9, 10, 11]: return 'autumn'
        elif month in [12, 1, 2]: return 'winter'
        return None

    df['계절'] = df['발생월'].apply(get_season)

    # ③ 계절별 건수 및 비율 계산
    seasonal_counts = df['계절'].value_counts().sort_index()
    seasonal_ratios = df['계절'].value_counts(normalize=True).sort_index()

    seasonal_summary = pd.DataFrame({
        '건수': seasonal_counts,
        '비율': seasonal_ratios
    })

    print("✅ 최근 5년간 계절별 산사태 발생 건수 및 비율:")
    for season in ['spring', 'summer', 'autumn', 'winter']:
        count = seasonal_summary.loc[season, '건수'] if season in seasonal_summary.index else 0
        ratio = seasonal_summary.loc[season, '비율'] if season in seasonal_summary.index else 0
        print(f"  - {season:<6}: {count:4d}건  ({ratio * 100:.2f}%)")

    return seasonal_summary

csv_path = "/content/산림청_최근 5년간 전국 산사태 발생 이력_20231114 (1).csv"
seasonal_summary = extract_landslide_season_ratio(csv_path)

✅ 최근 5년간 계절별 산사태 발생 건수 및 비율:
  - spring:    2건  (0.05%)
  - summer: 3211건  (78.32%)
  - autumn:  887건  (21.63%)
  - winter:    0건  (0.00%)


이때 winter의 산사태 발생 비율이 0.0000인 것을 확인할 수 있음. 빈도까지 0으로 나온것으로 보아 가중치를 0으로 설정하는게 타당할 것으로 판단.

### 먼저, 가중치를 고려한 5개년의 총 강수량 맵

In [ ]:
# 📦 개선된 강수량 Feature 분석 및 보간 코드 (Barnes 보간 적용 + 가중치 기반 총 강수량 포함, Polygon 기반 SHP 저장 + Spatial Join)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ✅ 계절별 산사태 발생 비율 기반 가중치
SEASONAL_WEIGHTS = {
    'spring': 0.0005,
    'summer': 0.7832,
    'autumn': 0.2163
}

# ✅ CSV 로드 함수
def try_read_csv(path):
    for enc in ['cp949', 'utf-8-sig', 'euc-kr', 'utf-8']:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            continue
    return None

def discover_csv_files():
    result = {}
    for year in range(2019, 2024):
        folder = f"/content/{year}년도"
        for month in range(1, 13):
            path = f"{folder}/{month}.csv"
            if not os.path.exists(path): continue
            df = try_read_csv(path)
            if df is None: continue
            col_map = {col: '경도' if 'lon' in col.lower() or '경도' in col.lower() else
                             '위도' if 'lat' in col.lower() or '위도' in col.lower() else
                             '강수량(mm)' if '강수' in col.lower() or 'rain' in col.lower() else col
                       for col in df.columns}
            df.rename(columns=col_map, inplace=True)
            if not {'경도', '위도', '강수량(mm)'}.issubset(df.columns): continue
            df.dropna(subset=['경도', '위도', '강수량(mm)'], inplace=True)
            result[f"{year}_{month:02d}"] = df
    return result

# ✅ 경상북도 경계 로드 함수
def load_gyeongbuk_boundary():
    path = "/content/AREA/경상북도.shp"
    if not os.path.exists(path): return None, None
    for enc in ['cp949', 'utf-8', 'euc-kr']:
        try:
            gdf = gpd.read_file(path, encoding=enc)
            return gdf.to_crs('EPSG:4326'), gdf.total_bounds
        except:
            continue
    return None, None

# ✅ 보간 처리 함수 (Barnes 보간법 적용)
from scipy.interpolate import griddata

def run_rainfall_analysis_for_features(boundary_gdf, monthly_data):
    seasonal_grids = {'spring': [], 'summer': [], 'autumn': []}

    minx, miny, maxx, maxy = boundary_gdf.total_bounds
    grid_x, grid_y = np.meshgrid(
        np.linspace(minx, maxx, 300),
        np.linspace(miny, maxy, 300)
    )
    grid_points = np.column_stack((grid_x.ravel(), grid_y.ravel()))
    inside_mask = [boundary_gdf.unary_union.contains(Point(x, y)) for x, y in grid_points]

    for key, df in monthly_data.items():
        year, month = map(int, key.split('_'))
        if month in [3, 4, 5]:
            season = 'spring'
        elif month in [6, 7, 8]:
            season = 'summer'
        elif month in [9, 10, 11]:
            season = 'autumn'
        else:
            continue

        lons = df['경도'].values
        lats = df['위도'].values
        rain = df['강수량(mm)'].values

        # ⚠️ Barnes 대신 griddata 적용
        grid_z = griddata(
            points=np.column_stack((lons, lats)),
            values=rain,
            xi=(grid_x, grid_y),
            method='linear'  # 또는 'cubic', 'nearest'
        )

        seasonal_grids[season].append(grid_z)

    seasonal_means = {}
    for season, grid_list in seasonal_grids.items():
        if grid_list:
            seasonal_means[season] = {'mean': np.nanmean(grid_list, axis=0)}

    return {
        'seasonal_grids': seasonal_means,
        'grid_x': grid_x,
        'grid_y': grid_y,
        'inside_mask': inside_mask
    }


# ✅ 임도망도 보간 결과 기반 계절 가중 강수량 계산 및 시각화 + Shapefile 저장

def compute_weighted_rainfall_grid(seasonal_grids):
    weighted_sum = None
    for season, weight in SEASONAL_WEIGHTS.items():
        if season not in seasonal_grids:
            continue
        if weighted_sum is None:
            weighted_sum = seasonal_grids[season]['mean'] * weight
        else:
            weighted_sum += seasonal_grids[season]['mean'] * weight
    return weighted_sum

def save_weighted_rainfall_png(grid, output_path):
    plt.figure(figsize=(10, 6))
    plt.imshow(grid, cmap='YlOrRd', origin='lower')
    plt.colorbar(label='가중치 적용 강수량 (mm)')
    plt.title('계절 가중 총 강수량')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

def save_weighted_rainfall_polygon_shp(grid, grid_x, grid_y, inside_mask, output_path):
    inside_mask_2d = np.array(inside_mask).reshape(grid.shape)
    polygons = []

    for i in range(grid.shape[0] - 1):
        for j in range(grid.shape[1] - 1):
            if not inside_mask_2d[i, j]: continue
            value = grid[i, j]
            if np.isnan(value): continue

            poly = Polygon([
                (grid_x[i, j], grid_y[i, j]),
                (grid_x[i+1, j], grid_y[i+1, j]),
                (grid_x[i+1, j+1], grid_y[i+1, j+1]),
                (grid_x[i, j+1], grid_y[i, j+1]),
                (grid_x[i, j], grid_y[i, j])
            ])
            polygons.append({"geometry": poly, "weighted_rainfall": value})

    gdf = gpd.GeoDataFrame(polygons, crs="EPSG:4326")
    gdf.to_file(output_path, encoding='utf-8')
    print(f"✅ Polygon 기반 SHP 저장 완료: {output_path}")

# ✅ Spatial Join 함수

def spatial_join_with_forest_road(weighted_shp_path, road_shp_path, output_path):
    rainfall_gdf = gpd.read_file(weighted_shp_path).to_crs("EPSG:4326")
    road_gdf = gpd.read_file(road_shp_path).to_crs("EPSG:4326")

    joined = gpd.sjoin(road_gdf, rainfall_gdf, how="left", predicate="intersects")
    joined.to_file(output_path, encoding="utf-8")
    print(f"✅ Spatial Join 완료: {output_path}")

# ✅ 전체 실행 함수

def generate_weighted_rainfall_outputs(seasonal_grids, grid_x, grid_y, inside_mask, output_dir):
    print("🌧️ 계절 가중 총 강수량 산출 중...")
    weighted_grid = compute_weighted_rainfall_grid(seasonal_grids)

    os.makedirs(output_dir, exist_ok=True)
    png_path = os.path.join(output_dir, "weighted_rainfall_map.png")
    shp_path = os.path.join(output_dir, "weighted_rainfall_polygon.shp")

    save_weighted_rainfall_png(weighted_grid, png_path)
    save_weighted_rainfall_polygon_shp(weighted_grid, grid_x, grid_y, inside_mask, shp_path)

    print(f"✅ PNG 저장 완료: {png_path}")

    # 🚧 Spatial Join 추가 실행
    forest_path = "/content/DATA/버퍼데이터10.shp"
    join_output = os.path.join(output_dir, "forest_roads_with_rainfall.shp")
    spatial_join_with_forest_road(shp_path, forest_path, join_output)

    return weighted_grid

# 🔄 메인 실행
if __name__ == '__main__':
    print("🛣️  임도망도 Feature용 강수량 분석을 시작합니다...")
    boundary_gdf, _ = load_gyeongbuk_boundary()
    monthly_data = discover_csv_files()

    if boundary_gdf is not None and monthly_data:
        results = run_rainfall_analysis_for_features(boundary_gdf, monthly_data)
        generate_weighted_rainfall_outputs(
            seasonal_grids=results['seasonal_grids'],
            grid_x=results['grid_x'],
            grid_y=results['grid_y'],
            inside_mask=results['inside_mask'],
            output_dir="./output_rainfall_features"
        )
    else:
        print("❌ 데이터 로드에 실패했습니다.")


🛣️  임도망도 Feature용 강수량 분석을 시작합니다...


### 고민할 부분
- 현재의 계절성을 고려한 지역별 총 강수량으로만 feature를 두는게 신뢰성있는 데이터분석이라고 할 수 있나?
- 일 별 데이터를 현재 사용하지 않았으므로 이것또한 따로 고려해야하는 feature로 볼 것인지(현재 보고서에 적혀있는 강수량 총합만을 고려한 상태) -> 이유는, 일별 강수량을 통해 며칠간의 집중호우 등의 이벤트를 고려하지 못하는 상태가 된다.
- 일별로 강우 강도 및 누적 강우량을 고려해야할 것 같음.(단시간 시계열 분석)

### 임도망도 데이터와의 JOIN 시
- 나중에 spatial join시에 각 포인트의 위치를 정확하게 구분해야함(어떤 포인트가 어떤행인지추적이 어려움)
- Barnes 보간 후에 발생하는 결측치 제거 및 마스킹
- 위험도 등급 필드 추가(시각적으로 총 강수량에 따른 class를 지정해두었을 때 시각적으로 위험도에 따른 강조가 가능함)

-> 현재 임도망도 데이터 사용해 join 시도중

### 적용부분
- point -> polygon 저장으로 전환

/content/AREA

In [16]:
!pip install metpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.3/424.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 22.1 MB/s eta 0:00:00
